
## Background + justification of design choices

##add use of LLM

### YOLOv8-Inspired Model Overview

We based our model architecture on YOLOv8, an anchor-free object detection system developed by Ultralytics. As a preliminary validation step, we loaded a pretrained YOLOv8 model and evaluated it on our training set. Given its strong performance in this proof-of-concept test, we proceeded to adapt the architecture for our task.

YOLOv8 (You Only Look Once, version 8) is an object detector that performs both object localization and classification in a single forward pass. For each detected object, it outputs a tensor containing the normalized bounding box coordinates, an objectness score, and class probabilities over a predefined set of categories.

The model is built on a convolutional neural network (CNN), which is well-suited for our dataset given its relatively limited size, especially compared to transformer-based architectures. Moreover, the deep layers of the YOLOv8 backbone enable hierarchical feature extraction across multiple spatial resolutions, which we expect allows the model to capture both the overall shape and finer surface textures of individual chocolates.

A key design feature of YOLOv8 is its anchor-free detection head, which simplifies training and enhances generalization by removing the need for manually designed anchor boxes. This allows the model to directly regress bounding box coordinates without relying on pre-defined priors, making it more flexible across diverse image conditions.

YOLOv8 consists of three main components:
- A CNN backbone based on a CSPDarknet variant with C2f modules, responsible for initial feature extraction,
- A neck that fuses features across different spatial scales,
- And an anchor-free detection head that outputs bounding box coordinates, objectness scores, and class probabilities.

We tried three different models inspired by this YOLOv8 architecture in our approach: YOLOv8Lite (our final submission), CompactYOLOv2, and TinyYOLO.[explain their differences briefly]

**Sources:**

Ultralytics YOLOv8 GitHub Repository: https://github.com/ultralytics/ultralytics

Ultralytics YOLOv8 Documentation: https://docs.ultralytics.com/

Ahmed, K., Yasir, A., & Rho, S. (2023). A Comprehensive Review of YOLO Architectures in Computer Vision: From YOLOv1 to YOLOv8 and YOLO-NAS. arXiv preprint arXiv:2304.00501. https://arxiv.org/abs/2304.00501

## Data and Preprocessing - change this
The training and testing datasets were annotated using Roboflow, where bounding boxes with class labels were added.

The training data were then augmented using rotations, scalings, jittering, Gaussian blurs, Gaussian noise, as well as manually added black occlusions to mimic the headband obstructions present in some training and validation images (see images below).

These occlusion augmentations were added after it was observed that the CNN model struggled particularly with segmenting and classifying chocolates obstructed by headbands. The goal was to provide the model with more representative examples to improve its robustness to such occlusions. Examples of images with black line occlusions are shown below.
The remaining augmentations were intended to mimic natural variations that may occur in the dataset.

Following all augmentations, including on the reference images, the final augmented training set comprised 740 images.



<img src="./Images%20for%20the%20report/L1010019_JPG.rf.9334969cee0225ae92b99869669bbda5_modified.jpg" width="200"/>
<img src="./Images%20for%20the%20report/L1010032_JPG.rf.92054891399871ff6ef0d683ea39630d_modified.jpg" width="200"/>



## Technical Description

### YOLOv8Lite

Below is a schematic of the YOLOv8Lite model, which is a lighter version for YOLOv8. Batchsize is taken to be 1.

<img src="./Images%20for%20the%20report/diagram_yolov8lite.jpeg" width="700"/>

Much like YOLOv8, our model YOLOv8Lite has three main components: a Stem (CNN backbone), a Neck (for feature fusion), an anchor-free Head (for detection) and ouputs the bounding boxes, objectness, and class scores for each image: (batch_size, anchors, 5 + num_classes, height, width). However, our head is single-scale (only one ConvBlock) while YOLOv8 has three.

ConvBlock is a building block for the CNN stem, which extracts the features by progressively downsampling the image while increasing channels, and is identical to YOLO v8's ConvModule.

The neck aggregates features using the SPPF module, which is a lightweight version YOLO v8's Spatial Pyramid Pooling – Fast module's structure. Its purpose is to fuse multiscale information without increasing the feature map size. 

The C2F Module is somewhat inspired by YOLOv8's C2f (Cross-Stage Partial with Feature concatenation) block in its
general structure and feature concatenation, but is very simplified; for instance, no skip connections. While in larger
models this may cause some issues with vanishing gradients and earlier layers of the network not receiving a strong
gradient signal, we didn't find this to be an issue during our training. However, we do note that this might be a
limitation when trying to scale our implementation.

The choice of SiLU activation was directly from YOLO v8, for which SiLU is the default. The SiLU activation function is
one of the attempts to have a continuous version of the ReLU activation. The SiLU is defined as follows:

$$
\operatorname{silu}(x) = x\sigma(x), \; \text{where } \sigma(x) \text{ is the logisitc sigmoid}
$$

One of the downsides of the ReLU activation function is the non-differentiability at the point $x=0$. At this point one
must make use of sub-gradients in order to perform optimization. However, it isn't clear which sub-gradient to choose
since any gradient in the range $[0, 1]$ would be correct. However, in non-convex optimization, sub-gradient methods
often display poor performance. Assume that we have some non-convex function $f$ with the following conditions:
 - $\lVert g \rVert_2 \leq G \quad \forall g \in \partial f$ where $\partial f$ is a sub-gradient of $f$
 - $\lVert x^{0} - x^{\star} \rVert_2 \leq R$
 - We choose the step-size as:
 $$
    \alpha_{k} = \frac{R}{G\sqrt{k}}
 $$
 then the iterates generated by the sub-gradient method satisfy:
$$
    \min_{0\leq i \leq k}f(x^{i}) - f(x^{\star}) \leq \frac{RG}{\sqrt{k}}
$$
In this case $x^{0}$ is the initialization and $x^{\star}$ is the optimal point. Note that condition (1) is
automatically satisfied when $f$ is G-Lipschitz. In actuality, this exact convergence rate cannot be achieved since we
cannot compute $R$ and it is computationally unfeasible to compute $G$. So we can only hope for a convergence rate of
$\mathcal{O}(1/\sqrt{k})$ where $k$ is the number of iterations. 

When using stochastic subgradient methods we have the following convergence in expectation guarantees:
$$
    \mathbb{E}\left[f(x^k) - f(x^{\star})\right] \leq \left( \frac{D^2}{\gamma_0} + \gamma_0M^2\right)\frac{2+\log k}{\sqrt{k}}
$$

However, when we have a fully differentiable function it is possible to achieve a $\mathcal{O}(\frac{1}{k})$ convergence
rate. Hence, it is in our best interest to have a fully differentiable activation function. The Ultralytics team also
state that the non-monotonicity of the SiLU function might help the model learn more complex patterns. Also since the
gradient of the SiLU function is non-zero near $x=0$ it can help avoid vanishing gradients in the case where the
pre-activation outputs are normalized or are close to 0.

Overall, our model has the core components and principles of yolov8, but is single-scale, single-head and simplified. We
choose to make these simplifications due to our limited dataset, relatively low number of classes and our limited
compute capabilities. We prioritized a model that was fast to train on our limited hardware.



## Qualitative Evaluation

We evaluated the model's performance after training on data with augmentations.

### General Observations
The model performed well overall, producing high-confidence predictions (typically between 0.8 and 1.0) for most chocolate types. Occasionally, for challenging images, the prediction confidences can be as low as 0.2.
Bounding boxes were generally accurate and well-placed, though they often did not cover the full extent of the chocolates. This partial coverage did not appear to negatively affect classification performance.

<img src="./Images%20for%20the%20report/BasicAugm%20Qualitative%20Analysis/img.png" width="300"/>

### Background Influence
The model handled patterned and cluttered backgrounds surprisingly well, provided the chocolates were not obscured.

However, highly distracting backgrounds, such as the cover of textbooks, led to occasional missed detections or false positives, particularly for chocolates with low contrast against the background. For instance, Jelly Milk was sometimes falsely detected in background regions with similar color.

Overall, performance declined when chocolates blended into the background, highlighting the importance of contrast for effective segmentation. Small false positives also occurred when background colors resembled certain chocolates.

<img src="./Images%20for%20the%20report/BasicAugm%20Qualitative%20Analysis/output10.png" width="300"/>

### Occlusion and Object Interference
Black headbands did not impair detection as long as they did not obscure the chocolates. However, when obstructions covered part of a chocolate, missed detections became common.

Similarly, the presence of additional objects in the image did not affect performance unless they overlapped with the chocolates or cast strong shadows over them.

<img src="./Images%20for%20the%20report/BasicAugm%20Qualitative%20Analysis/output12.png" width="300"/>
<img src="./Images%20for%20the%20report/BasicAugm%20Qualitative%20Analysis/output11.png" width="300"/>
<img src="./Images%20for%20the%20report/BasicAugm%20Qualitative%20Analysis/output.png" width="300"/>

### Per-Class Performance
Amandinas were the most challenging: confidence scores sometimes dropped to ~0.2, likely due to their complex visual features—fine patterns and multiple colors.

Jelly Whites were occasionally missed when heavily shadowed.

In contrast, the model successfully distinguished visually similar pairs:
- Comtesse vs. Jelly White, suggesting strong shape recognition.
- Triangolo vs. Tentation Noir, indicating the model learned subtle differences in shape and color despite shared textures.